## **DCGAN: Deep Convolutional Generative Adversarial Networks**

Caso queira revisar a implementação veja esse post: [Understanding DCGANs: Deep Convolutional Generative Adversarial Networks
](https://medium.com/@danushidk507/understanding-dcgans-deep-convolutional-generative-adversarial-networks-1984bc028bf8)

Nesta aula vamos treinar uma DCGAN para gerar rostos 64×64 e investigar uma pergunta: **o que
acontece quando o ruído $z$ usado para gerar as imagens vem de uma distribuição diferente da usada
no treino?**

Vamos considerar três distribuições para $Z \sim p(z)$ — normal, uniforme e Weibull — e as
combinações abaixo, em que o gerador é treinado com uma e testado com outra:

| **$Z_{Train}$** | **$Z_{Test}$** |
| :-------------: | :------------: |
|      normal     |     uniform    |
|      normal     |     weibull    |
|     uniform     |     normal     |
|     uniform     |     weibull    |
|     weibull     |     normal     |
|     weibull     |     uniform    |

No fim da aula você vai conseguir prever o resultado de cada linha **sem precisar treinar seis
GANs** — é o Exercício 2.

> ⚡ **Esta aula precisa de GPU.** No Colab: *Ambiente de execução → Alterar o tipo de ambiente de
> execução → GPU (T4)*.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import numpy as np

import time
import os

In [ ]:
# Diagnóstico do ambiente — rode esta célula primeiro
import sys, sklearn

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA disp.  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print(">>> SEM GPU. O treino desta aula leva horas na CPU.")
    print(">>> Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)")

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)

### O ruído de entrada: três distribuições para $z$

O gerador transforma um vetor aleatório $z$ numa imagem. A função abaixo sorteia esse vetor com a
distribuição escolhida, e o gráfico mostra cada uma **exatamente como o gerador vai recebê-la**.

In [ ]:
def rand_vector(distribution, size, device="cpu"):
    """
    Gera vetores latentes z com a distribuição pedida.
    """
    if distribution == "normal":
        return torch.randn(size, device=device)
    elif distribution == "uniform":
        return (torch.rand(size, device=device) * 2 - 1)
    elif distribution == "weibull":
        q = torch.from_numpy(np.random.weibull(50, size)).float().to(device)
        q = (q - q.mean()) / q.std()
        return q
    else:
        raise ValueError("Specify a valid distribution: normal, uniform, weibull")


# As três distribuições, na mesma escala para dar para comparar
n = 1_000_000
plt.figure(figsize=(16, 4))
for k, nome in enumerate(["normal", "uniform", "weibull"]):
    amostras = rand_vector(nome, (n,)).numpy()
    plt.subplot(1, 3, k + 1)
    plt.hist(amostras, bins=200, range=(-7, 4), density=True)
    plt.title(f"{nome}   (média {amostras.mean():+.2f}, desvio {amostras.std():.2f})")
    plt.xlabel("valor de z")
    plt.ylabel("densidade")
plt.tight_layout()
plt.show()

### Carregando e preparando os dados:

Usamos o **Labeled Faces in the Wild (LFW)**: 13.233 fotos de rostos de pessoas públicas. O
scikit-learn baixa o dataset (cerca de 230 MB) na primeira vez e recorta cada foto para nós: um
quadrado em volta do rosto, reduzido para 64×64.

> ⏱️ **A primeira execução da célula abaixo leva alguns minutos**, por causa do download e da
> leitura das 13 mil fotos. Depois disso o resultado fica em cache e a célula roda em segundos.

In [ ]:
# ==========================
# Hyperparameters
# ==========================
LATENT_DIM = 64      # tamanho do vetor z
IMG_SIZE = 64        # as redes abaixo assumem imagens 64x64
BATCH_SIZE = 32
EPOCHS = 100
LEAKY_SLOPE = 0.2
LR = 1e-4
N_IMAGENS = 3000     # quantas fotos do LFW usar no treino (o dataset tem 13.233)

DISTRIBUTION_train = 'normal'
DISTRIBUTION_test  = 'weibull'

# ==========================
# Dataset
# ==========================
from sklearn.datasets import fetch_lfw_people

# `slice_` recorta um quadrado de 125x125 pixels em volta do rosto; `resize` leva a 64x64.
lfw = fetch_lfw_people(color=True, resize=IMG_SIZE / 125,
                       slice_=(slice(70, 195), slice(62, 187)),
                       min_faces_per_person=0)


class FacesDataset(Dataset):
    def __init__(self, imagens):
        self.imagens = imagens

    def __len__(self):
        return len(self.imagens)

    def __getitem__(self, idx):
        image = self.imagens[idx]              # (H, W, C), valores em [0, 1]
        image = image * 2 - 1                  # normalize to [-1, 1]
        # (H, W, C) → (C, H, W)
        return torch.from_numpy(image).permute(2, 0, 1)


escolhidas = np.random.permutation(len(lfw.images))[:N_IMAGENS]
dataset = FacesDataset(lfw.images[escolhidas].astype(np.float32))
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Dataset: {len(dataset)} imagens (de {len(lfw.images)} disponíveis)")

# ==========================
# Visualização
# ==========================
def plot_images(batch, n=8):
    """Exibe as n primeiras imagens de um batch normalizado em [-1, 1]."""
    imgs = batch[:n].detach().cpu().permute(0, 2, 3, 1).numpy()
    imgs = ((imgs + 1) * 127.5).clip(0, 255).astype(np.uint8)
    plt.figure(figsize=(2 * n, 2))
    for i, img in enumerate(imgs):
        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.axis('off')
    plt.show()

# Testar visualização
sample = next(iter(dataloader))
print(f"Images shape: {tuple(sample.shape)}")
plot_images(sample)

### Criando as redes Gerador e Discriminador

In [ ]:
class Generator(nn.Module):
    def __init__(self, input_dim=128, leaky_slope=0.2):
        super(Generator, self).__init__()
        self.input_dim = input_dim
        self.leaky_slope = leaky_slope

        self.net = nn.Sequential(
            # FC: latent vector → feature map (4x4x1024)
            nn.Linear(input_dim, 1024 * 4 * 4),
            nn.LeakyReLU(leaky_slope, inplace=True),

            # reshape handled in forward()
            
            # 4x4 → 8x8
            nn.ConvTranspose2d(1024, 512, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(leaky_slope, inplace=True),
            nn.Dropout(0.1),

            # 8x8 → 16x16
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(leaky_slope, inplace=True),
            nn.Dropout(0.1),

            # 16x16 → 32x32
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(leaky_slope, inplace=True),
            nn.Dropout(0.1),

            # 32x32 → 64x64
            nn.ConvTranspose2d(256, 3, kernel_size=4, stride=2, padding=1),
            # nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1),
            # nn.LeakyReLU(leaky_slope, inplace=True),
            # nn.Dropout(0.1),

            # # 64x64 → 128x128
            # nn.ConvTranspose2d(64, 3, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.net[0](z)  # Linear
        x = self.net[1](x)  # LeakyReLU
        x = x.view(-1, 1024, 4, 4)  # Reshape latent → feature map
        x = self.net[2:](x)  # Pass through ConvTranspose blocks
        return x


generator = Generator(LATENT_DIM, LEAKY_SLOPE).to(device)
print(generator)

In [ ]:
# class Discriminator(nn.Module):
#     def __init__(self, leaky_slope=0.2):
#         super(Discriminator, self).__init__()
#         self.leaky_slope = leaky_slope

#         self.net = nn.Sequential(
#             # 128x128x3 → 64x64x64
#             nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(leaky_slope, inplace=True),

#             # 64x64x64 → 32x32x128
#             nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(leaky_slope, inplace=True),

#             # 32x32x128 → 16x16x256
#             nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(leaky_slope, inplace=True),

#             # 16x16x256 → 8x8x512
#             nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(leaky_slope, inplace=True),

#             nn.Flatten(),
#             nn.Linear(512 * 8 * 8, 1),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, leaky_slope=0.2):
        super(Discriminator, self).__init__()
        self.leaky_slope = leaky_slope

        self.net = nn.Sequential(
            # 64x64x3 → 32x32x64
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(leaky_slope, inplace=True),

            # 32x32x64 → 16x16x128
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(leaky_slope, inplace=True),

            # 16x16x128 → 8x8x256
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(leaky_slope, inplace=True),

            # 8x8x256 → 4x4x512
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(leaky_slope, inplace=True),

            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


discriminator = Discriminator(LEAKY_SLOPE).to(device)
print(discriminator)

### Criando a GAN

In [ ]:
class GAN(nn.Module):
    def __init__(self, generator, discriminator):
        super(GAN, self).__init__()
        self.generator = generator
        self.discriminator = discriminator

    def forward(self, z):
        # Generate fake images
        fake_imgs = self.generator(z)
        # Evaluate with discriminator
        validity = self.discriminator(fake_imgs)
        return validity

In [ ]:
# Create models
generator = Generator(LATENT_DIM, LEAKY_SLOPE).to(device)
discriminator = Discriminator(LEAKY_SLOPE).to(device)

# Loss
criterion = nn.BCELoss()

# Optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=LR, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.999))

# Create GAN wrapper (optional)
gan = GAN(generator, discriminator).to(device)

In [ ]:
output_dir = "resultados_gan"
os.makedirs(output_dir, exist_ok=True)


def plot_generated_images(id, generator, input_dist, samples=9, dim=(3,3), figsize=(5,5),
                          device="cpu", z=None):
    # Vetores latentes: sorteados na hora, ou os que forem passados em `z`
    if z is None:
        z = rand_vector(input_dist, (samples, LATENT_DIM), device=device)

    # Generate images
    generator.eval()
    with torch.no_grad():
        generated_images = generator(z)

    # Rescale from [-1,1] → [0,255] and convert to numpy
    images = ((generated_images + 1) * 127.5).clamp(0, 255).cpu().numpy().astype('uint8')

    # images shape: (samples, 3, 64, 64) → transpose to (samples, 64, 64, 3)
    images = images.transpose(0, 2, 3, 1)

    # Plot
    plt.figure(figsize=figsize)
    for i in range(len(images)):
        plt.subplot(dim[0], dim[1], i+1)
        plt.imshow(images[i])
        plt.axis('off')
    plt.suptitle(id)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'GAN_image_{id}.png'))
    plt.show()

## Exercício 1 — O passo de treino

O treino de uma GAN alterna dois passos, e **cada um deles atualiza só uma das redes**. Você vai
implementá-los.

#### O algoritmo

Em cada batch:

1. **Treinar o discriminador**

   * Gerar previsões de imagens falsas a partir do gerador
   * A referência é **1 para imagens reais** e **0 para imagens falsas**
   * Treinar o discriminador (**com o gerador congelado**)

2. **Treinar o gerador**

   * Congelar os pesos do discriminador
   * O modelo "gan" produz imagens falsas a partir de ruído aleatório e obtém a saída do discriminador. Treinar o modelo "gan" com o discriminador congelado
   * A referência é **1 para as imagens falsas** (ou seja, enganar o discriminador)

Complete as duas funções abaixo. Elas usam as variáveis já criadas: `generator`, `discriminator`,
`optimizer_G`, `optimizer_D`, `criterion`, `rand_vector`, `DISTRIBUTION_train` e `LATENT_DIM`.

**`passo_discriminador(real_batch)`** recebe um lote de imagens reais (já no `device`) e deve:

1. sortear $z$ e gerar um lote de imagens falsas do mesmo tamanho;
2. calcular a loss do discriminador nas reais (alvo **real**) e nas falsas (alvo **falso**), e
   tirar a média das duas;
3. atualizar **só o discriminador**;
4. devolver a loss.

**`passo_gerador(batch_size)`** deve:

1. congelar o discriminador;
2. sortear $z$, gerar imagens falsas e passá-las pelo discriminador;
3. calcular a loss com o alvo que faz o gerador **enganar** o discriminador;
4. atualizar **só o gerador**, descongelar o discriminador e devolver a loss.

Para os alvos do discriminador, use `alvo_real(n)` e `alvo_falso(n)`, definidos abaixo. Eles
aplicam *label smoothing*: em vez de exatamente 1 e 0, sorteiam valores um pouco abaixo de 1 e um
pouco acima de 0, o que deixa o discriminador menos confiante e o treino mais estável.

**Atenção a um erro muito comum:** "congelar" uma rede **não** é colocá-la em `.eval()`. O `.eval()`
só muda o comportamento de camadas como Dropout e BatchNorm, e não impede o cálculo de gradientes
(lembre da Aula 1). Congelar é com `requires_grad = False`. Por isso, **as duas redes devem ficar
em `.train()` nos dois passos** — comece cada função chamando `.train()` nas duas.

Duas perguntas para pensar enquanto implementa:

- no passo do discriminador, por que as imagens falsas precisam de `.detach()`?
- no passo do gerador, o gerador quer que o discriminador responda o quê?

In [ ]:
def alvo_real(n):
    return torch.ones(n, 1, device=device) - 0.05 * torch.rand(n, 1, device=device)


def alvo_falso(n):
    return torch.zeros(n, 1, device=device) + 0.05 * torch.rand(n, 1, device=device)

In [ ]:
def passo_discriminador(real_batch):
    # seu código aqui
    raise NotImplementedError("Complete o passo do discriminador")


def passo_gerador(batch_size):
    # seu código aqui
    raise NotImplementedError("Complete o passo do gerador")

#### resposta

In [ ]:
def passo_discriminador(real_batch):
    n = real_batch.size(0)
    generator.train()
    discriminator.train()

    # Imagens falsas. O `.detach()` corta o grafo: este passo só treina o
    # discriminador, então o backward não precisa (nem deve) atravessar o gerador.
    z = rand_vector(DISTRIBUTION_train, (n, LATENT_DIM), device=device)
    fake_batch = generator(z).detach()

    optimizer_D.zero_grad()
    loss_real = criterion(discriminator(real_batch), alvo_real(n))   # reais -> 1
    loss_fake = criterion(discriminator(fake_batch), alvo_falso(n))  # falsas -> 0
    d_loss = (loss_real + loss_fake) / 2
    d_loss.backward()
    optimizer_D.step()
    return d_loss


def passo_gerador(batch_size):
    generator.train()
    discriminator.train()

    # Congela o discriminador: ele só serve de "juiz" neste passo. Congelar é com
    # requires_grad; `.eval()` não congela nada (só muda Dropout e BatchNorm).
    for p in discriminator.parameters():
        p.requires_grad = False

    z = rand_vector(DISTRIBUTION_train, (batch_size, LATENT_DIM), device=device)
    optimizer_G.zero_grad()
    saida = discriminator(generator(z))
    # Alvo 1: o gerador é recompensado quando o discriminador diz "real".
    g_loss = criterion(saida, torch.ones(batch_size, 1, device=device))
    g_loss.backward()
    optimizer_G.step()

    for p in discriminator.parameters():
        p.requires_grad = True
    return g_loss

#### Testando os passos

A célula abaixo roda os seus dois passos num par de redes **novas** (as suas não são alteradas) e
confere, entre outras coisas, se cada passo atualiza só a rede que deveria. Quando um teste
falha, a mensagem diz o que provavelmente está errado. Só siga para o treino quando tudo passar.

In [ ]:
def _sem_grad(modulo):
    return all(p.grad is None for p in modulo.parameters())


def _copia(modulo):
    return [p.detach().clone() for p in modulo.parameters()]


def _mudou(antes, modulo):
    return any(not torch.equal(a, p.detach()) for a, p in zip(antes, modulo.parameters()))


def _ok(msg):
    print("  ✅", msg)


def testar_passos():
    """Confere os dois passos de treino num par de redes novas, sem mexer nas suas."""
    global generator, discriminator, optimizer_G, optimizer_D
    originais = (generator, discriminator, optimizer_G, optimizer_D)
    try:
        torch.manual_seed(0)
        generator = Generator(LATENT_DIM, LEAKY_SLOPE).to(device)
        discriminator = Discriminator(LEAKY_SLOPE).to(device)
        optimizer_G = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
        optimizer_D = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
        reais = next(iter(dataloader)).to(device)
        n = reais.size(0)

        print("passo_discriminador")
        g_antes, d_antes = _copia(generator), _copia(discriminator)
        generator.eval()
        discriminator.eval()
        loss = passo_discriminador(reais)
        assert torch.is_tensor(loss) and loss.dim() == 0 and torch.isfinite(loss), \
            "passo_discriminador deve devolver a loss: um tensor escalar e finito."
        _ok("devolve uma loss escalar e finita")
        assert generator.training and discriminator.training, (
            "Depois do passo, as duas redes deveriam estar em modo de treino (`.train()`). "
            "`.eval()` não congela a rede: ele só muda Dropout e BatchNorm, e alternar o modo "
            "entre os passos faz o discriminador julgar o gerador com um comportamento diferente "
            "daquele com que foi treinado.")
        _ok("deixa as duas redes em modo de treino")
        assert _sem_grad(generator), (
            "O passo do discriminador calculou gradientes para o GERADOR. Faltou o `.detach()` "
            "nas imagens falsas? Sem ele, o backward atravessa o gerador inteiro sem necessidade.")
        _ok("não calcula gradientes para o gerador")
        assert _mudou(d_antes, discriminator), \
            "Os pesos do discriminador não mudaram. Faltou `optimizer_D.step()`?"
        _ok("atualiza o discriminador")
        assert not _mudou(g_antes, generator), \
            "Os pesos do GERADOR mudaram no passo do discriminador. Só o discriminador deve ser atualizado."
        _ok("não mexe no gerador")

        # aprende a separar? (varios passos com o gerador parado)
        z = rand_vector(DISTRIBUTION_train, (n, LATENT_DIM), device=device)
        with torch.no_grad():
            generator.eval()
            falsas = generator(z)
            discriminator.eval()
            sep_antes = (discriminator(reais).mean() - discriminator(falsas).mean()).item()
        for _ in range(15):
            passo_discriminador(reais)
        with torch.no_grad():
            generator.eval()
            discriminator.eval()
            sep_depois = (discriminator(reais).mean() - discriminator(falsas).mean()).item()
        assert sep_depois > sep_antes + 0.05, (
            "Depois de vários passos, o discriminador não aprendeu a separar reais de falsas "
            f"(D(real) − D(falsa) foi de {sep_antes:+.3f} para {sep_depois:+.3f}). "
            "Os alvos estão certos? Real deve ser 1, falsa deve ser 0.")
        _ok(f"aprende a separar reais de falsas (D(real) − D(falsa): {sep_antes:+.2f} → {sep_depois:+.2f})")

        print("passo_gerador")
        for p in discriminator.parameters():
            p.grad = None
        generator.zero_grad(set_to_none=True)
        g_antes, d_antes = _copia(generator), _copia(discriminator)
        generator.eval()
        discriminator.eval()
        loss = passo_gerador(n)
        assert torch.is_tensor(loss) and loss.dim() == 0 and torch.isfinite(loss), \
            "passo_gerador deve devolver a loss: um tensor escalar e finito."
        _ok("devolve uma loss escalar e finita")
        assert generator.training and discriminator.training, (
            "Depois do passo, as duas redes deveriam estar em modo de treino (`.train()`). "
            "`.eval()` não congela a rede: ele só muda Dropout e BatchNorm, e alternar o modo "
            "entre os passos faz o discriminador julgar o gerador com um comportamento diferente "
            "daquele com que foi treinado.")
        _ok("deixa as duas redes em modo de treino")
        assert _sem_grad(discriminator), (
            "O passo do gerador calculou gradientes para o DISCRIMINADOR. Faltou congelá-lo "
            "(`requires_grad = False`) antes do backward?")
        _ok("não calcula gradientes para o discriminador")
        assert all(p.requires_grad for p in discriminator.parameters()), (
            "O discriminador continua congelado depois do passo do gerador. Descongele no fim, "
            "senão o próximo passo do discriminador não aprende nada.")
        _ok("descongela o discriminador no fim")
        assert _mudou(g_antes, generator), \
            "Os pesos do gerador não mudaram. Faltou `optimizer_G.step()`?"
        _ok("atualiza o gerador")
        assert not _mudou(d_antes, discriminator), \
            "Os pesos do DISCRIMINADOR mudaram no passo do gerador. Só o gerador deve ser atualizado."
        _ok("não mexe no discriminador")

        # engana cada vez mais? (varios passos com o discriminador parado)
        z = rand_vector(DISTRIBUTION_train, (256, LATENT_DIM), device=device)
        with torch.no_grad():
            generator.eval()
            discriminator.eval()
            antes = discriminator(generator(z)).mean().item()
        for _ in range(15):
            passo_gerador(n)
        with torch.no_grad():
            generator.eval()
            discriminator.eval()
            depois = discriminator(generator(z)).mean().item()
        assert depois > antes + 0.02, (
            "Depois de vários passos do gerador, o discriminador não passou a achar as falsas "
            f"mais reais (D(G(z)) foi de {antes:.3f} para {depois:.3f}). "
            "O alvo da loss do gerador está certo? O gerador quer que o discriminador responda 'real'.")
        _ok(f"aprende a enganar o discriminador (D(G(z)): {antes:.2f} → {depois:.2f})")

        print("\n🎉 Tudo certo! Pode rodar o treino.")
    finally:
        generator, discriminator, optimizer_G, optimizer_D = originais


testar_passos()

## Treinando a GAN

> ⏱️ **O treino é longo:** com `EPOCHS = 100`, algo como 10 minutos numa T4. Enquanto ele roda,
> adiante o **Exercício 3**: não dá para executar células com o treino em andamento, mas dá para
> escrever o código. Se o tempo estiver curto, `EPOCHS = 30` já produz rostos reconhecíveis.
>
> As figuras de progresso usam **sempre o mesmo $z$**. Assim, o que muda de uma figura para a
> outra é só o gerador aprendendo — e fica fácil perceber se ele passar a produzir o mesmo rosto
> para todos os $z$.

In [ ]:
d_loss_history = []
g_loss_history = []

# O mesmo z em todas as figuras de progresso
z_fixo = rand_vector(DISTRIBUTION_train, (9, LATENT_DIM), device=device)

start_time = time.time()

for e in range(1, EPOCHS+1):
    soma_d, soma_g = 0.0, 0.0

    for real_batch in dataloader:
        real_batch = real_batch.to(device)
        d_loss = passo_discriminador(real_batch)
        g_loss = passo_gerador(real_batch.size(0))
        soma_d += d_loss.item()
        soma_g += g_loss.item()

    # Record losses (média da época)
    d_loss_history.append(soma_d / len(dataloader))
    g_loss_history.append(soma_g / len(dataloader))

    # Print epoch info
    print(f"Epoch: {e}, Time: {time.time() - start_time:.2f}s, "
          f"g_loss: {g_loss_history[-1]:.4f}, d_loss: {d_loss_history[-1]:.4f}")

    # Salva os pesos e mostra o progresso na época 1, a cada 10 e na última
    if e == 1 or e % 10 == 0 or e == EPOCHS:
        torch.save(generator.state_dict(), os.path.join(output_dir, f'generator_epoch{e}.pt'))
        torch.save(discriminator.state_dict(), os.path.join(output_dir, f'discriminator_epoch{e}.pt'))
        plot_generated_images(f'epoca_{e}', generator, DISTRIBUTION_train, device=device, z=z_fixo)

# Learning curves
plt.figure(figsize=(6,5))
plt.plot(d_loss_history, color='blue', label='Discriminator')
plt.plot(g_loss_history, color='red', label='Generator')
plt.legend()
plt.grid(True)
plt.title('Learning curves')
plt.ylabel('Loss (média da época)')
plt.xlabel('Epochs')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "Learning_curves.png"), dpi=150)
plt.show()

### Teste

Carregamos o gerador salvo na última época e geramos imagens com a distribuição usada no treino e
com a distribuição de teste.

In [ ]:
caminho = os.path.join(output_dir, f'generator_epoch{EPOCHS}.pt')
assert os.path.exists(caminho), f"Não encontrei {caminho}. Rode o treino da seção anterior primeiro."

generator = Generator(LATENT_DIM, LEAKY_SLOPE).to(device)
generator.load_state_dict(torch.load(caminho, map_location=device))
generator.eval()

In [ ]:
# Generate images with training distribution
plot_generated_images(
    f'Tr_{DISTRIBUTION_train}---Ts_{DISTRIBUTION_train}',
    generator,
    DISTRIBUTION_train,
    samples=16,
    dim=(4,4),
    figsize=(6,6),
    device=device
)

# Generate images with test distribution
plot_generated_images(
    f'Tr_{DISTRIBUTION_train}---Ts_{DISTRIBUTION_test}',
    generator,
    DISTRIBUTION_test,
    samples=16,
    dim=(4,4),
    figsize=(6,6),
    device=device
)

## Exercício 2 — Explorando o espaço latente

O gerador aprendeu a transformar vetores $z$ em rostos. Neste exercício você vai investigar
**como a posição de $z$ no espaço latente afeta a imagem** — e, com isso, responder à pergunta do
início da aula sem precisar treinar seis GANs.

As funções abaixo já estão prontas: `gerar` passa um lote de vetores pelo gerador, `mostrar_linha`
desenha um lote de imagens lado a lado, e `plotar_curvas` desenha os gráficos que você vai
preencher.

In [ ]:
def gerar(z):
    generator.eval()
    with torch.no_grad():
        return generator(z)


def mostrar_linha(imgs, titulo=""):
    imgs = ((imgs + 1) / 2).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
    plt.figure(figsize=(1.6 * len(imgs), 1.9))
    for i, img in enumerate(imgs):
        plt.subplot(1, len(imgs), i + 1)
        plt.imshow(img)
        plt.axis("off")
    plt.suptitle(titulo, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


def plotar_curvas(escalas, div, sat, pontos=None):
    if len(div) != len(escalas) or len(sat) != len(escalas):
        print("⚠️  As listas ainda não têm um valor para cada escala — complete o laço da 2.2.")
        return
    ref_div, ref_sat = diversidade(reais), saturacao(reais)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    for ax, ys, ref, nome in [(a1, div, ref_div, "diversidade"), (a2, sat, ref_sat, "saturação")]:
        ax.plot(escalas, ys, "o-", label="gerador, z = escala × ε")
        ax.axhline(ref, color="gray", ls="--", label="imagens reais")
        for dist, (desvio, d, s) in (pontos or {}).items():
            y = d if nome == "diversidade" else s
            ax.plot(desvio, y, "s", ms=10, label=f"z {dist}")
        ax.set_xlabel("escala (desvio-padrão de z)")
        ax.set_ylabel(nome)
        ax.grid(True)
        ax.legend()
    plt.tight_layout()
    plt.show()


# Material fixo para todos os itens
torch.manual_seed(0)
eps8 = torch.randn(8, LATENT_DIM, device=device)      # 8 "direções" para as figuras
eps512 = torch.randn(512, LATENT_DIM, device=device)  # 512 para as medidas
reais = torch.stack([dataset[i] for i in range(512)]).to(device)

### 2.1 — Duas medidas

Olhar imagens é subjetivo. Implemente duas medidas simples para um lote de imagens
`imgs` de shape `(N, 3, 64, 64)`, com valores em $[-1, 1]$:

- **`diversidade(imgs)`**: o quanto as imagens do lote diferem **entre si**. Calcule o
  desvio-padrão de cada pixel **ao longo do lote** e tire a média. Um lote de imagens idênticas
  deve dar 0.
- **`saturacao(imgs)`**: a fração dos pixels "estourados", com valor absoluto acima de 0,98.
  Serve para detectar artefatos.

As duas devem devolver um `float`.

In [ ]:
def diversidade(imgs):
    # seu código aqui
    raise NotImplementedError("Complete diversidade")


def saturacao(imgs):
    # seu código aqui
    raise NotImplementedError("Complete saturacao")

#### resposta

In [ ]:
def diversidade(imgs):
    # std ao longo do LOTE (dim=0): um desvio para cada pixel de cada canal
    return imgs.std(dim=0).mean().item()


def saturacao(imgs):
    return (imgs.abs() > 0.98).float().mean().item()

#### Conferindo as medidas

In [ ]:
iguais = (torch.rand(1, 3, 64, 64) * 2 - 1).repeat(10, 1, 1, 1)
assert isinstance(diversidade(iguais), float), "diversidade deve devolver um float (use .item())."
assert abs(diversidade(iguais)) < 1e-6, (
    "Um lote de imagens IDÊNTICAS deve ter diversidade 0, mas deu "
    f"{diversidade(iguais):.4f}. O desvio-padrão é ao longo do lote (dim=0)?")
diferentes = torch.rand(10, 3, 64, 64) * 2 - 1
assert diversidade(diferentes) > 0.3, "Imagens aleatórias diferentes deveriam ter diversidade alta."
assert abs(saturacao(torch.full((2, 3, 4, 4), 0.99)) - 1.0) < 1e-6, "Pixels em 0,99 estão saturados."
assert abs(saturacao(torch.full((2, 3, 4, 4), -0.99)) - 1.0) < 1e-6, "Pixels em −0,99 também estão."
assert saturacao(torch.zeros(2, 3, 4, 4)) == 0.0, "Pixels em 0 não estão saturados."
print("✅ Medidas conferidas.")
print(f"Imagens reais: diversidade {diversidade(reais):.3f}, saturação {saturacao(reais):.4f}")

### 2.2 — Mexendo na escala de $z$

O gerador foi treinado com $z \sim \mathcal{N}(0, 1)$. Vamos gerar imagens a partir de
$z = s \cdot \varepsilon$, com $\varepsilon \sim \mathcal{N}(0, 1)$ fixo e a escala $s$ variando.

**Antes de rodar, escreva a sua previsão** numa célula de texto: o que você espera ver com
$s = 0$? E com $s = 0{,}5$? E com $s = 3$?

Depois, complete a célula abaixo. Para cada escala:

1. mostre uma linha com as imagens geradas a partir de `s * eps8` — as mesmas 8 direções em todas
   as linhas, para dar para comparar;
2. gere as imagens de `s * eps512` e guarde a diversidade e a saturação delas nas listas.

In [ ]:
escalas = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
div_escala, sat_escala = [], []

# seu código aqui

plotar_curvas(escalas, div_escala, sat_escala)

#### resposta

In [ ]:
escalas = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
div_escala, sat_escala = [], []

for s in escalas:
    mostrar_linha(gerar(s * eps8), f"escala {s}")
    imgs = gerar(s * eps512)
    div_escala.append(diversidade(imgs))
    sat_escala.append(saturacao(imgs))

plotar_curvas(escalas, div_escala, sat_escala)

**O que aparece.** Com $s = 0$ todas as colunas viram **o mesmo rosto**: $z = 0$ é um único ponto,
e o gerador devolve uma espécie de "rosto médio". Conforme $s$ cresce, os rostos se diferenciam e
ganham contraste. Em $s = 1$, a escala do treino, a diversidade do gerador fica praticamente igual
à das imagens reais. Acima disso a diversidade continua subindo — mas não porque os rostos ficaram
mais variados, e sim porque estouraram: é o que a curva de saturação mostra.

Note também que **cada coluna mantém a mesma "pessoa"** em todas as escalas. A *direção* de $z$
decide quem é o rosto; o *tamanho* de $z$ decide o quão típico ou exagerado ele sai.

Esse controle tem nome: é o **truncation trick**, usado por modelos como BigGAN e StyleGAN.
Reduzir a escala de $z$ troca diversidade por fidelidade — imagens mais "seguras", menos
variadas.

### 2.3 — De volta à tabela do início

Agora gere 512 imagens com cada uma das três distribuições da aula (`"normal"`, `"uniform"`,
`"weibull"`, usando `rand_vector`). Para cada uma, guarde no dicionário `pontos` uma tupla com
**o desvio-padrão do `z` usado**, a diversidade e a saturação. O gráfico vai marcar cada
distribuição na posição do seu desvio-padrão.

Depois responda: **use o gráfico para prever o que acontece em cada linha da tabela do início
da aula.** Quais trocas devem estragar as imagens? Quais quase não devem mudar nada? O formato da
distribuição importa, ou só a escala?

In [ ]:
pontos = {}   # nome da distribuição -> (desvio de z, diversidade, saturação)

# seu código aqui

plotar_curvas(escalas, div_escala, sat_escala, pontos=pontos)

#### resposta

In [ ]:
pontos = {}
for nome in ["normal", "uniform", "weibull"]:
    z = rand_vector(nome, (512, LATENT_DIM), device=device)
    imgs = gerar(z)
    pontos[nome] = (z.std().item(), diversidade(imgs), saturacao(imgs))
    print(f"{nome:8s} desvio de z {z.std().item():.3f}   "
          f"diversidade {pontos[nome][1]:.3f}   saturação {pontos[nome][2]:.4f}   "
          f"(z entre {z.min().item():+.2f} e {z.max().item():+.2f})")

plotar_curvas(escalas, div_escala, sat_escala, pontos=pontos)

**As três distribuições caem em cima da curva de escala.** A uniforme de `rand_vector` vai de
$-1$ a $1$, e por isso tem desvio-padrão $1/\sqrt{3} \approx 0{,}58$ — ela **não** foi
padronizada, ao contrário da Weibull. Então, para o gerador, "trocar normal por uniforme" é
basicamente "encolher $z$ para 58% do tamanho".

A Weibull é o contraexemplo que prova o ponto: o formato dela é bem diferente da normal (é torta,
com cauda longa para a esquerda e um corte à direita), mas como foi padronizada para desvio 1, ela
gera imagens com **a mesma diversidade e a mesma saturação** da normal.

Com isso dá para prever a tabela toda:

| Treino → Teste | O que muda na escala | Resultado esperado |
|---|---|---|
| normal → uniform | 1 → 0,58 | rostos mais parecidos entre si, apagados; sem artefatos |
| normal → weibull | 1 → 1 | quase nenhuma diferença |
| uniform → normal | 0,58 → 1 (1,7×) | rostos estourados, com artefatos |
| uniform → weibull | 0,58 → 1 (1,7×) | igual ao anterior |
| weibull → normal | 1 → 1 | quase nenhuma diferença |
| weibull → uniform | 1 → 0,58 | rostos mais parecidos, apagados |

Treinando de fato os três geradores (100 épocas cada) e medindo, a previsão se confirma. A
variação em relação a testar com a própria distribuição de treino foi:

| Treino → Teste | Δ diversidade | Δ saturação |
|---|---|---|
| normal → uniform | −0,104 | −0,011 |
| normal → weibull | +0,001 | −0,001 |
| uniform → normal | +0,129 | +0,039 |
| uniform → weibull | +0,127 | +0,036 |
| weibull → normal | −0,006 | −0,001 |
| weibull → uniform | −0,105 | −0,005 |

Repare em *uniform → normal* e *uniform → weibull*: duas distribuições de formatos bem diferentes,
e praticamente o mesmo efeito — porque as duas têm desvio 1.

**A conclusão é que, aqui, a escala domina e o formato quase não importa.** Se as três
distribuições tivessem sido padronizadas para o mesmo desvio, a tabela inteira seria
"quase nenhuma diferença".

> **Para casa:** treine com `DISTRIBUTION_train = 'uniform'` e confira a previsão para
> *uniform → normal* com os seus próprios olhos.

### 2.4 — Andando entre dois rostos

Implemente `interpolar(z1, z2, n)`: devolve `n` vetores indo de `z1` a `z2` em linha reta,
com o primeiro igual a `z1` e o último igual a `z2`. O resultado deve ter shape
`(n, LATENT_DIM)`.

Depois rode a célula de baixo e olhe **as normas** impressas. O que acontece com a norma no meio
do caminho? Por quê? Relacione com o que você viu na 2.2.

In [ ]:
def interpolar(z1, z2, n):
    # seu código aqui
    raise NotImplementedError("Complete interpolar")

#### resposta

In [ ]:
def interpolar(z1, z2, n):
    t = torch.linspace(0, 1, n, device=z1.device).view(-1, 1)
    return (1 - t) * z1 + t * z2

#### Andando no espaço latente

In [ ]:
torch.manual_seed(7)
z1 = torch.randn(1, LATENT_DIM, device=device)
z2 = torch.randn(1, LATENT_DIM, device=device)
caminho = interpolar(z1, z2, 9)

assert caminho.shape == (9, LATENT_DIM), f"Shape esperado (9, {LATENT_DIM}), veio {tuple(caminho.shape)}."
assert torch.allclose(caminho[0], z1[0]) and torch.allclose(caminho[-1], z2[0]), \
    "O primeiro vetor deve ser z1 e o último deve ser z2."

mostrar_linha(gerar(caminho), "de z1 a z2")
print("normas ao longo do caminho:", " ".join(f"{v:.2f}" for v in caminho.norm(dim=1).tolist()))

#### Desafio (opcional)

1. Implemente a **interpolação esférica** (*slerp*) e compare as normas com as da linear.
2. Calcule a nota média que o **discriminador** dá às imagens de cada escala da 2.2
   (`discriminator(imgs).mean()`). Você esperaria que ela medisse a qualidade das imagens.
   Mede? Por quê?

#### resposta

In [ ]:
def slerp(z1, z2, n):
    t = torch.linspace(0, 1, n, device=z1.device).view(-1, 1)
    cos = (z1 / z1.norm() * z2 / z2.norm()).sum().clamp(-1, 1)
    omega = torch.acos(cos)
    return (torch.sin((1 - t) * omega) * z1 + torch.sin(t * omega) * z2) / torch.sin(omega)


caminho_s = slerp(z1, z2, 9)
mostrar_linha(gerar(caminho_s), "slerp")
print("normas (linear):", " ".join(f"{v:.2f}" for v in caminho.norm(dim=1).tolist()))
print("normas (slerp) :", " ".join(f"{v:.2f}" for v in caminho_s.norm(dim=1).tolist()))

# nota do discriminador por escala
discriminator.eval()
for s in escalas:
    with torch.no_grad():
        nota = discriminator(gerar(s * eps512)).mean().item()
    print(f"escala {s:4.2f}: D(G(z)) médio = {nota:.3f}")

**Interpolação.** Na linear, a norma cai no meio do caminho. Para dois vetores gaussianos
independentes, o ponto médio $\tfrac{1}{2}(z_1 + z_2)$ tem norma de cerca de $1/\sqrt{2} \approx 0{,}71$
da norma das pontas: os dois vetores são quase ortogonais em 64 dimensões, e somar metade de
cada um "encurta" o vetor. Pela 2.2, o meio do caminho equivale a usar uma escala menor — um
rosto um pouco mais genérico. A slerp anda sobre a esfera e mantém a norma constante, por isso é
a interpolação preferida para $z$ gaussiano. Nesta resolução a diferença visual é pequena; nos
números ela é clara.

**O discriminador não é uma medida de qualidade.** A nota varia pouco, não acompanha a
qualidade das imagens e nem sempre vai na mesma direção: dependendo do treino, as imagens
estouradas de $s = 3$ podem receber nota **maior** que as boas, de $s = 1$. Repare também que até as
imagens reais recebem uma nota modesta, entre 0,5 e 0,6. Dois motivos:

- no equilíbrio do treino, o discriminador responde perto de 0,5 para quase tudo: é exatamente o
  que significa o gerador estar conseguindo enganá-lo;
- ele só foi treinado para separar as imagens reais das que o gerador produz com $s = 1$. Imagens
  de $s = 3$ são algo que ele **nunca viu**, e a resposta dele ali não quer dizer nada.

É por isso que se avaliam GANs com métricas externas, como a FID, que compara estatísticas das
imagens geradas com as das reais usando uma rede treinada separadamente.

## Exercício 3 — GAN condicional

A DCGAN de rostos gera um rosto qualquer: não dá para pedir *"um homem de óculos"*. Uma **GAN
condicional** resolve isso dando ao gerador, além de $z$, uma informação sobre **o que** gerar —
aqui, a classe da imagem.

Vamos usar o **Fashion-MNIST**, que vocês já conhecem da Aula 1: 28×28, um canal, 10 classes, e
treina rápido. O objetivo é poder pedir *"gere uma bota"* e receber uma bota.

A ideia tem duas partes, e as duas são obrigatórias:

- **o gerador recebe a classe**: ele gera a partir de $(z, \text{classe})$;
- **o discriminador também recebe a classe**: ele julga o **par** (imagem, classe). Sem isso, o
  gerador poderia devolver uma bota perfeita quando pedimos uma camiseta, e o discriminador não
  teria como reclamar.

Se quiser uma referência, o tutorial
"[Rede Adversarial Generativa Condicional](https://www.geeksforgeeks.org/deep-learning/conditional-generative-adversarial-network/)"
implementa a mesma ideia em TensorFlow.

In [ ]:
from torchvision import datasets, transforms

LATENT_C = 100
N_CLASSES = 10
NOMES = ["camiseta", "calça", "pulôver", "vestido", "casaco",
         "sandália", "camisa", "tênis", "bolsa", "bota"]

fmnist = datasets.FashionMNIST(
    root="data", train=True, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]),
)
loader_c = DataLoader(fmnist, batch_size=128, shuffle=True, drop_last=True)

imgs, classes = next(iter(loader_c))
print(f"imagens {tuple(imgs.shape)} em [{imgs.min():.0f}, {imgs.max():.0f}]   classes {tuple(classes.shape)}")

### 3.1 — As duas redes

Complete as duas classes. Uma sugestão de arquitetura, que funciona bem:

**`GeradorCondicional(z, classes)`** → imagens `(N, 1, 28, 28)` em $[-1, 1]$

- `nn.Embedding(N_CLASSES, 50)` transforma cada classe num vetor de 50 números;
- concatene esse vetor com `z` (fica com `LATENT_C + 50` números);
- `Linear` para `128 × 7 × 7`, `BatchNorm1d`, `ReLU`, e reformate para `(N, 128, 7, 7)`;
- `ConvTranspose2d(128, 64, 4, 2, 1)` → 14×14, `BatchNorm2d`, `ReLU`;
- `ConvTranspose2d(64, 1, 4, 2, 1)` → 28×28, `Tanh`.

**`DiscriminadorCondicional(imagens, classes)`** → probabilidade `(N, 1)` de o par ser real

- `nn.Embedding(N_CLASSES, 28 * 28)` transforma cada classe num "mapa" que, reformatado, vira
  `(N, 1, 28, 28)`;
- concatene esse mapa com a imagem **como um segundo canal**: `(N, 2, 28, 28)`;
- `Conv2d(2, 64, 4, 2, 1)` → 14×14, `LeakyReLU(0.2)`;
- `Conv2d(64, 128, 4, 2, 1)` → 7×7, `BatchNorm2d`, `LeakyReLU(0.2)`;
- `Flatten`, `Linear` para 1, `Sigmoid`.

As classes chegam como um tensor de inteiros de shape `(N,)`.

In [ ]:
class GeradorCondicional(nn.Module):
    def __init__(self, latent_dim=LATENT_C, n_classes=N_CLASSES):
        super().__init__()
        # seu código aqui

    def forward(self, z, classes):
        # seu código aqui
        raise NotImplementedError("Complete o GeradorCondicional")


class DiscriminadorCondicional(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        # seu código aqui

    def forward(self, imagens, classes):
        # seu código aqui
        raise NotImplementedError("Complete o DiscriminadorCondicional")

#### resposta

In [ ]:
class GeradorCondicional(nn.Module):
    def __init__(self, latent_dim=LATENT_C, n_classes=N_CLASSES):
        super().__init__()
        self.emb = nn.Embedding(n_classes, 50)
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + 50, 128 * 7 * 7),
            nn.BatchNorm1d(128 * 7 * 7),
            nn.ReLU(True),
        )
        self.conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 7x7 -> 14x14
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1),     # 14x14 -> 28x28
            nn.Tanh(),
        )

    def forward(self, z, classes):
        # O condicionamento inteiro está nesta linha: o gerador recebe z E a classe.
        x = torch.cat([z, self.emb(classes)], dim=1)
        x = self.fc(x).view(-1, 128, 7, 7)
        return self.conv(x)


class DiscriminadorCondicional(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.emb = nn.Embedding(n_classes, 28 * 28)
        self.net = nn.Sequential(
            nn.Conv2d(2, 64, 4, 2, 1),              # 28x28 -> 14x14
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),            # 14x14 -> 7x7
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),
        )

    def forward(self, imagens, classes):
        # A classe vira um "mapa" 28x28 e entra como segundo canal da imagem:
        # o discriminador julga o PAR (imagem, classe), não só a imagem.
        mapa = self.emb(classes).view(-1, 1, 28, 28)
        return self.net(torch.cat([imagens, mapa], dim=1))

### 3.2 — Os passos de treino

São os mesmos do Exercício 1, com uma diferença: **o discriminador julga pares
(imagem, classe)**.

- `passo_d_cond(imagens, classes)`: recebe imagens reais com as classes verdadeiras. Para as
  falsas, sorteie classes com `torch.randint(0, N_CLASSES, (n,), device=device)`, gere com elas, e
  passe **essas mesmas classes** ao discriminador.
- `passo_g_cond(batch_size)`: sorteie $z$ e classes, gere, e tente fazer o discriminador aceitar o
  par (imagem gerada, classe pedida) como real.

As redes e os otimizadores se chamam `gen_c`, `disc_c`, `opt_gc` e `opt_dc`. Use
`torch.randn(n, LATENT_C, device=device)` para $z$, e `alvo_real` / `alvo_falso` do Exercício 1
para os alvos.

In [ ]:
def passo_d_cond(imagens, classes):
    # seu código aqui
    raise NotImplementedError("Complete o passo do discriminador condicional")


def passo_g_cond(batch_size):
    # seu código aqui
    raise NotImplementedError("Complete o passo do gerador condicional")

#### resposta

In [ ]:
def passo_d_cond(imagens, classes):
    n = imagens.size(0)
    gen_c.train()
    disc_c.train()

    # Falsas geradas com classes sorteadas; o discriminador recebe essas MESMAS classes.
    z = torch.randn(n, LATENT_C, device=device)
    classes_falsas = torch.randint(0, N_CLASSES, (n,), device=device)
    falsas = gen_c(z, classes_falsas).detach()

    opt_dc.zero_grad()
    loss_real = criterion(disc_c(imagens, classes), alvo_real(n))
    loss_fake = criterion(disc_c(falsas, classes_falsas), alvo_falso(n))
    d_loss = (loss_real + loss_fake) / 2
    d_loss.backward()
    opt_dc.step()
    return d_loss


def passo_g_cond(batch_size):
    gen_c.train()
    disc_c.train()
    for p in disc_c.parameters():
        p.requires_grad = False

    z = torch.randn(batch_size, LATENT_C, device=device)
    classes = torch.randint(0, N_CLASSES, (batch_size,), device=device)
    opt_gc.zero_grad()
    # O gerador quer que o discriminador aceite o par (imagem gerada, classe pedida).
    saida = disc_c(gen_c(z, classes), classes)
    g_loss = criterion(saida, torch.ones(batch_size, 1, device=device))
    g_loss.backward()
    opt_gc.step()

    for p in disc_c.parameters():
        p.requires_grad = True
    return g_loss

#### Testando as redes e os passos

In [ ]:
def testar_condicional():
    """Confere as redes e os passos condicionais em modelos novos, sem mexer nos seus."""
    global gen_c, disc_c, opt_gc, opt_dc
    originais = tuple(globals().get(k) for k in ("gen_c", "disc_c", "opt_gc", "opt_dc"))
    try:
        torch.manual_seed(0)
        gen_c = GeradorCondicional().to(device)
        disc_c = DiscriminadorCondicional().to(device)
        assert len(list(gen_c.parameters())) > 0 and len(list(disc_c.parameters())) > 0, \
            "As redes não têm parâmetros. Faltou criar as camadas no __init__?"
        opt_gc = optim.Adam(gen_c.parameters(), lr=2e-4, betas=(0.5, 0.999))
        opt_dc = optim.Adam(disc_c.parameters(), lr=2e-4, betas=(0.5, 0.999))

        print("as redes")
        z = torch.randn(8, LATENT_C, device=device)
        classes = torch.arange(8, device=device) % N_CLASSES
        gen_c.eval()
        disc_c.eval()
        with torch.no_grad():
            img = gen_c(z, classes)
            assert tuple(img.shape) == (8, 1, 28, 28), \
                f"O gerador devolveu shape {tuple(img.shape)}; o esperado é (8, 1, 28, 28)."
            # com um z enorme, só uma saída limitada (Tanh) continua dentro de [-1, 1]
            extremo = gen_c(z * 100, classes)
            assert img.min() >= -1 and img.max() <= 1 and extremo.abs().max() <= 1, \
                "As imagens geradas podem sair de [-1, 1]. Faltou o Tanh no fim?"
            _ok("gerador: (z, classes) → imagens (N, 1, 28, 28) em [-1, 1]")

            p = disc_c(img, classes)
            assert tuple(p.shape) == (8, 1), \
                f"O discriminador devolveu shape {tuple(p.shape)}; o esperado é (8, 1)."
            assert p.min() >= 0 and p.max() <= 1, \
                "A saída do discriminador saiu de [0, 1]. Faltou o Sigmoid no fim?"
            _ok("discriminador: (imagens, classes) → probabilidades (N, 1)")

            zero = torch.zeros(8, dtype=torch.long, device=device)
            nove = torch.full((8,), 9, dtype=torch.long, device=device)
            dif_g = (gen_c(z, zero) - gen_c(z, nove)).abs().mean().item()
            assert dif_g > 1e-4, (
                "O gerador ignora a classe: para o mesmo z, pedir uma camiseta ou uma bota dá "
                "exatamente a mesma imagem. A classe precisa entrar no forward.")
            _ok("o gerador usa a classe")
            dif_d = (disc_c(img, zero) - disc_c(img, nove)).abs().mean().item()
            assert dif_d > 1e-5, (
                "O discriminador ignora a classe: a mesma imagem recebe a mesma nota com qualquer "
                "classe. Sem isso ele não consegue punir 'uma bota rotulada como camiseta'.")
            _ok("o discriminador usa a classe")

        imgs, cls = next(iter(loader_c))
        imgs, cls = imgs.to(device), cls.to(device)
        n = imgs.size(0)

        print("passo_d_cond")
        g_antes, d_antes = _copia(gen_c), _copia(disc_c)
        gen_c.eval()
        disc_c.eval()
        loss = passo_d_cond(imgs, cls)
        assert torch.is_tensor(loss) and loss.dim() == 0 and torch.isfinite(loss), \
            "passo_d_cond deve devolver a loss: um tensor escalar e finito."
        assert gen_c.training and disc_c.training, (
            "Depois do passo, as duas redes deveriam estar em modo de treino (`.train()`). "
            "`.eval()` não congela a rede: ele só muda Dropout e BatchNorm, e alternar o modo "
            "entre os passos faz o discriminador julgar o gerador com um comportamento diferente "
            "daquele com que foi treinado.")
        _ok("deixa as duas redes em modo de treino")
        assert _sem_grad(gen_c), \
            "O passo do discriminador calculou gradientes para o gerador. Faltou o `.detach()`?"
        _ok("não calcula gradientes para o gerador")
        assert _mudou(d_antes, disc_c) and not _mudou(g_antes, gen_c), \
            "O passo do discriminador deve atualizar o discriminador, e só ele."
        _ok("atualiza só o discriminador")

        print("passo_g_cond")
        for p in disc_c.parameters():
            p.grad = None
        g_antes, d_antes = _copia(gen_c), _copia(disc_c)
        gen_c.eval()
        disc_c.eval()
        loss = passo_g_cond(n)
        assert torch.is_tensor(loss) and loss.dim() == 0 and torch.isfinite(loss), \
            "passo_g_cond deve devolver a loss: um tensor escalar e finito."
        assert gen_c.training and disc_c.training, (
            "Depois do passo, as duas redes deveriam estar em modo de treino (`.train()`). "
            "`.eval()` não congela a rede: ele só muda Dropout e BatchNorm, e alternar o modo "
            "entre os passos faz o discriminador julgar o gerador com um comportamento diferente "
            "daquele com que foi treinado.")
        _ok("deixa as duas redes em modo de treino")
        assert _sem_grad(disc_c), \
            "O passo do gerador calculou gradientes para o discriminador. Faltou congelá-lo?"
        _ok("não calcula gradientes para o discriminador")
        assert all(p.requires_grad for p in disc_c.parameters()), \
            "O discriminador continua congelado depois do passo do gerador. Descongele no fim."
        _ok("descongela o discriminador no fim")
        assert _mudou(g_antes, gen_c) and not _mudou(d_antes, disc_c), \
            "O passo do gerador deve atualizar o gerador, e só ele."
        _ok("atualiza só o gerador")

        print("aprendizado")
        z = torch.randn(n, LATENT_C, device=device)
        cls_f = torch.randint(0, N_CLASSES, (n,), device=device)

        # medido em modo de treino: com BatchNorm, as estatísticas acumuladas ainda não
        # se estabilizaram depois de poucos passos
        def separacao():
            gen_c.train()
            disc_c.train()
            with torch.no_grad():
                return (disc_c(imgs, cls).mean() - disc_c(gen_c(z, cls_f), cls_f).mean()).item()

        antes = separacao()
        for _ in range(15):
            passo_d_cond(imgs, cls)
        depois = separacao()
        assert depois > antes + 0.05, (
            f"O discriminador não aprendeu a separar reais de falsas ({antes:+.3f} → {depois:+.3f}). "
            "Confira os alvos: pares reais → 1, pares falsos → 0.")
        _ok(f"o discriminador aprende a separar ({antes:+.2f} → {depois:+.2f})")

        def enganacao():
            gen_c.train()
            disc_c.train()
            with torch.no_grad():
                return disc_c(gen_c(z, cls_f), cls_f).mean().item()

        antes = enganacao()
        for _ in range(15):
            passo_g_cond(n)
        depois = enganacao()
        assert depois > antes + 0.02, (
            f"O gerador não aprendeu a enganar o discriminador ({antes:.3f} → {depois:.3f}). "
            "O alvo da loss do gerador deve ser 'real' (1).")
        _ok(f"o gerador aprende a enganar ({antes:.2f} → {depois:.2f})")

        print("\n🎉 Tudo certo! Pode treinar a GAN condicional.")
    finally:
        for k, v in zip(("gen_c", "disc_c", "opt_gc", "opt_dc"), originais):
            if v is None:
                globals().pop(k, None)
            else:
                globals()[k] = v


testar_condicional()

### 3.3 — Treinando e pedindo imagens

A célula abaixo treina a GAN condicional e mostra uma grade: **cada linha é uma classe** e
**cada coluna usa o mesmo $z$**. O treino leva alguns minutos numa T4.

In [ ]:
EPOCHS_C = 10

torch.manual_seed(0)
gen_c = GeradorCondicional().to(device)
disc_c = DiscriminadorCondicional().to(device)
assert list(gen_c.parameters()) and list(disc_c.parameters()),     "As redes condicionais ainda estão vazias. Complete a 3.1 e passe no teste da 3.2 antes de treinar."
opt_gc = optim.Adam(gen_c.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_dc = optim.Adam(disc_c.parameters(), lr=2e-4, betas=(0.5, 0.999))

z_grade = torch.randn(10, LATENT_C, device=device)


def grade_condicional(titulo):
    gen_c.eval()
    with torch.no_grad():
        z = z_grade.repeat(N_CLASSES, 1)                               # linha i, coluna j -> z_j
        cls = torch.arange(N_CLASSES, device=device).repeat_interleave(10)  # linha i -> classe i
        imgs = gen_c(z, cls).cpu()
    fig, eixos = plt.subplots(N_CLASSES, 10, figsize=(9, 10))
    for i in range(N_CLASSES):
        for j in range(10):
            eixos[i, j].imshow(imgs[i * 10 + j, 0], cmap="gray", vmin=-1, vmax=1)
            eixos[i, j].set_xticks([])
            eixos[i, j].set_yticks([])
        eixos[i, 0].set_ylabel(NOMES[i], rotation=0, ha="right", va="center", fontsize=9)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()


inicio = time.time()
for e in range(1, EPOCHS_C + 1):
    soma_d = soma_g = 0.0
    for imgs, cls in loader_c:
        imgs, cls = imgs.to(device), cls.to(device)
        soma_d += passo_d_cond(imgs, cls).item()
        soma_g += passo_g_cond(imgs.size(0)).item()
    print(f"época {e:2d}: d_loss {soma_d / len(loader_c):.3f}  "
          f"g_loss {soma_g / len(loader_c):.3f}  ({time.time() - inicio:.0f}s)")
    if e in (1, EPOCHS_C):
        grade_condicional(f"Época {e}: cada linha é uma classe, cada coluna usa o mesmo z")

### 3.4 — Lendo a grade

Responda olhando a grade da última época:

1. **Olhe uma linha.** As 10 imagens são da mesma classe? São todas iguais, ou variam?
2. **Olhe uma coluna.** O $z$ é o mesmo nas 10 imagens. Existe algo em comum entre elas, mesmo
   sendo de classes diferentes? O que o $z$ parece controlar, e o que a classe controla?
3. **Tente você mesmo:** escreva uma célula que gera 8 **bolsas** diferentes.

#### resposta

In [ ]:
gen_c.eval()
with torch.no_grad():
    z = torch.randn(8, LATENT_C, device=device)
    bolsas = gen_c(z, torch.full((8,), NOMES.index("bolsa"), device=device))

plt.figure(figsize=(12, 1.8))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(bolsas[i, 0].cpu(), cmap="gray", vmin=-1, vmax=1)
    plt.axis("off")
plt.suptitle("8 bolsas")
plt.show()

1. **Cada linha é uma classe só**, e as imagens variam entre si: são tênis diferentes, vestidos
   diferentes. A variação vem do $z$.
2. **Nas colunas, o $z$ controla o "estilo"**, e **a classe controla o conteúdo**. O efeito mais
   fácil de ver é o brilho: há colunas em que quase todas as peças saem escuras, e outras em que
   quase todas saem claras, qualquer que seja a classe. Em algumas colunas dá para notar também
   um padrão de formato. Isso se chama *desemaranhamento* (*disentanglement*): informações
   diferentes controladas por entradas diferentes. Ele raramente é perfeito — e a grade de vocês
   pode mostrar isso com mais ou menos clareza.
3. Basta fixar a classe e variar o $z$. Compare com a DCGAN de rostos: lá, o único controle que
   temos é o próprio $z$, e ninguém sabe de antemão qual $z$ dá um rosto de óculos.

## Avaliação

Deixe seu feedback da aula pelo **formulário linkado na coluna _Feedback_ do
[README do repositório](https://github.com/Erickslb/deep-learning-fgv-2026)** — o que funcionou, o que ficou confuso, o que faltou.

Dúvidas técnicas que podem interessar aos colegas ficam melhor como
[issue](https://github.com/Erickslb/deep-learning-fgv-2026/issues), que todo mundo vê.